# SISTEMA COMPLETO DE DETECCIÓN Y MEDICIÓN DE PECES
Incluye: Entrenamiento K-Fold + Validación Externa + Medición Física
Autor: Ignacio Rehbein
Universidad San Sebastián — ICIF H001 — Inteligencia Artificial


In [1]:
from ultralytics import YOLO
import os
import torch
import math
import cv2
import argparse

print('Verificando entorno y dispositivo...')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo activo: {DEVICE.upper()}')

BASE_PATH   = os.getcwd()
IMAGES_PATH = os.path.join(BASE_PATH, 'images')
RUNS_PATH   = os.path.join(BASE_PATH, 'runs_kfold')
VAL_IMAGES  = os.path.join(IMAGES_PATH, 'val')
OUTPUT_DIR  = os.path.join(BASE_PATH, 'val_final')
MODEL_BASE  = 'yolov8s.pt'
EPOCHS      = 20
IMG_SIZE    = 640
BATCH_SIZE  = 8

os.makedirs(RUNS_PATH, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Directorios configurados correctamente.')


Verificando entorno y dispositivo...
Dispositivo activo: CUDA
Directorios configurados correctamente.


## Entrenamiento K-Fold (5 folds)

In [ ]:
def run_kfold(base_path, model_base, epochs, img_size, batch_size):
    print('Entrenando con validación cruzada K-Fold (5 folds)')
    folds = [1, 2, 3, 4, 5]

    for val_fold in folds:
        print(f'FOLD {val_fold}: Validando con fold{val_fold}, entrenando con los demás')
        train_folds = [f'images/fold{i}' for i in folds if i != val_fold]
        val_fold_path = f'images/fold{val_fold}'

        data_yaml = os.path.join(base_path, f'data_fold{val_fold}.yaml')
        with open(data_yaml, 'w', encoding='utf-8') as f:
            f.write(f'''# Dataset YOLOv8 - Fold {val_fold}
path: {os.path.abspath(base_path).replace(os.sep, '/')}
train:
{chr(10).join(['  - ' + fold for fold in train_folds])}
val: {val_fold_path}
names:
  0: fish
''')

        model = YOLO(model_base)
        results = model.train(
            data=data_yaml,
            epochs=epochs,
            imgsz=img_size,
            batch=batch_size,
            project=os.path.join(base_path, 'runs_kfold'),
            name=f'fold{val_fold}_train',
            exist_ok=True,
            augment=True
        )

        trained_model = YOLO(os.path.join(results.save_dir, 'weights', 'best.pt'))
        output_dir = os.path.join(base_path, f'val{val_fold}')
        os.makedirs(output_dir, exist_ok=True)
        trained_model.predict(
            source=val_fold_path,
            conf=0.25,
            imgsz=img_size,
            save=True,
            project=output_dir,
            name='',
            exist_ok=True
        )

        print(f'Fold {val_fold} completado. Resultados guardados en {output_dir}')

run_kfold(BASE_PATH, MODEL_BASE, EPOCHS, IMG_SIZE, BATCH_SIZE)


Entrenando con validación cruzada K-Fold (5 folds)
FOLD 1: Validando con fold1, entrenando con los demás
New https://pypi.org/project/ultralytics/8.3.221 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.0  Python-3.12.0 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3060 Laptop GPU, 6144MiB)
engine\trainer: task=detect, mode=train, model=yolov8s.pt, data=C:\Users\ignac\Desktop\a-main\data_fold1.yaml, epochs=20, time=None, patience=100, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=C:\Users\ignac\Desktop\a-main\runs_kfold, name=fold1_train, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=Fal

train: Scanning C:\Users\ignac\Desktop\a-main\labels\fold2... 1054 images, 118 backgrounds, 0 corrupt: 100%|██████████| 1054/1054 [00:00<00:00, 1229.34it/s]


train: New cache created: C:\Users\ignac\Desktop\a-main\labels\fold2.cache


val: Scanning C:\Users\ignac\Desktop\a-main\labels\fold1... 263 images, 20 backgrounds, 0 corrupt: 100%|██████████| 263/263 [00:00<00:00, 979.97it/s]


val: New cache created: C:\Users\ignac\Desktop\a-main\labels\fold1.cache
Plotting labels to C:\Users\ignac\Desktop\a-main\runs_kfold\fold1_train\labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.002, momentum=0.9) with parameter groups 63 weight(decay=0.0), 70 weight(decay=0.0005), 69 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to C:\Users\ignac\Desktop\a-main\runs_kfold\fold1_train
Starting training for 20 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      15/20      2.21G      1.186      0.596      1.256          6        640: 100%|██████████| 132/132 [00:30<00:00,  4.30it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 17/17 [00:02<00:00,  5.97it/s]

                   all        263        337      0.946      0.933      0.975      0.658



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


      16/20      2.22G      1.135     0.5744      1.196         11        640:  30%|███       | 40/132 [00:09<00:21,  4.29it/s]

## Validación final externa

In [4]:
def find_best_model(base_path):
    for root, _, files in os.walk(base_path):
        for file in files:
            if file == 'best.pt':
                return os.path.join(root, file)
    return None

def validate_final_external():
    print('Buscando el mejor modelo en los folds...')
    best_model = find_best_model(RUNS_PATH)
    if not best_model:
        print('No se encontró ningún modelo best.pt en runs_kfold.')
        return
    print(f'Modelo seleccionado: {best_model}')
    model = YOLO(best_model)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    results = model.predict(
        source=VAL_IMAGES,
        conf=0.25,
        imgsz=IMG_SIZE,
        device=DEVICE,
        save=True,
        project=OUTPUT_DIR,
        name='',
        exist_ok=True
    )
    print(f'Validación externa completada. Resultados guardados en: {OUTPUT_DIR}')

validate_final_external()


Buscando el mejor modelo en los folds...
Modelo seleccionado: C:\Users\ignac\Desktop\a-main\runs_kfold\fold1_train\weights\best.pt

image 1/263 C:\Users\ignac\Desktop\a-main\images\val\seq_0001_00255.jpeg: 480x640 1 fish, 7.6ms
image 2/263 C:\Users\ignac\Desktop\a-main\images\val\seq_0001_00256.jpeg: 480x640 1 fish, 7.2ms
image 3/263 C:\Users\ignac\Desktop\a-main\images\val\seq_0001_00257.jpeg: 480x640 1 fish, 6.9ms
image 4/263 C:\Users\ignac\Desktop\a-main\images\val\seq_0001_00258.jpeg: 480x640 1 fish, 7.3ms
image 5/263 C:\Users\ignac\Desktop\a-main\images\val\seq_0001_00262.jpeg: 480x640 1 fish, 7.0ms
image 6/263 C:\Users\ignac\Desktop\a-main\images\val\seq_0001_00263.jpeg: 480x640 1 fish, 6.9ms
image 7/263 C:\Users\ignac\Desktop\a-main\images\val\seq_0001_00264.jpeg: 480x640 1 fish, 6.9ms
image 8/263 C:\Users\ignac\Desktop\a-main\images\val\seq_0001_00265.jpeg: 480x640 1 fish, 6.9ms
image 9/263 C:\Users\ignac\Desktop\a-main\images\val\seq_0001_00266.jpeg: 480x640 1 fish, 6.9ms
imag

## Medición física de peces

In [5]:
MODEL_PATH = find_best_model(RUNS_PATH)
VAL_IMAGES_DIR = VAL_IMAGES
OUTPUT_MEASURED = os.path.join(BASE_PATH, 'val_medidos')

BANDEJA_CM = 50.0
BANDEJA_PX = 1456.0
FACTOR_CM = BANDEJA_CM / BANDEJA_PX

os.makedirs(OUTPUT_MEASURED, exist_ok=True)
print(f'Factor de conversión: {FACTOR_CM:.5f} cm/px')

if not MODEL_PATH:
    raise FileNotFoundError('No se encontró el modelo entrenado para medición.')

model = YOLO(MODEL_PATH)
print(f'Modelo cargado: {MODEL_PATH}')

for img_name in os.listdir(VAL_IMAGES_DIR):
    if not img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue

    img_path = os.path.join(VAL_IMAGES_DIR, img_name)
    img = cv2.imread(img_path)
    results = model(img, verbose=False, device=DEVICE)

    for r in results:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            class_name = model.names[cls]

            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            w, h = abs(x2 - x1), abs(y2 - y1)
            ratio = w / h if h else 0

            if ratio > 1.3:
                cy = (y1 + y2) // 2
                cv2.line(img, (x1, cy), (x2, cy), (255, 0, 0), 2)
                longitud_px = w
            elif ratio < 0.7:
                cx = (x1 + x2) // 2
                cv2.line(img, (cx, y1), (cx, y2), (255, 0, 0), 2)
                longitud_px = h
            else:
                cv2.line(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
                longitud_px = math.hypot(x2 - x1, y2 - y1)

            longitud_cm = longitud_px * FACTOR_CM
            mid_x, mid_y = (x1 + x2) // 2, (y1 + y2) // 2
            cv2.putText(img, f'{longitud_cm:.1f} cm', (mid_x - 60, mid_y - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
            cv2.putText(img, f'{class_name} ({conf:.2f})', (x1, y2 + 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200, 255, 0), 2)

    out_path = os.path.join(OUTPUT_MEASURED, img_name)
    cv2.imwrite(out_path, img)
    print(f'{img_name} medida y guardada ({longitud_cm:.1f} cm aprox.)')

print('Mediciones completadas correctamente (valores en centímetros).')


Factor de conversión: 0.03434 cm/px
Modelo cargado: C:\Users\ignac\Desktop\a-main\runs_kfold\fold1_train\weights\best.pt
seq_0001_00255.jpeg medida y guardada (12.6 cm aprox.)
seq_0001_00256.jpeg medida y guardada (13.6 cm aprox.)
seq_0001_00257.jpeg medida y guardada (11.1 cm aprox.)
seq_0001_00258.jpeg medida y guardada (9.0 cm aprox.)
seq_0001_00262.jpeg medida y guardada (6.9 cm aprox.)
seq_0001_00263.jpeg medida y guardada (9.3 cm aprox.)
seq_0001_00264.jpeg medida y guardada (14.7 cm aprox.)
seq_0001_00265.jpeg medida y guardada (14.2 cm aprox.)
seq_0001_00266.jpeg medida y guardada (14.5 cm aprox.)
seq_0001_00267.jpeg medida y guardada (14.8 cm aprox.)
seq_0001_00268.jpeg medida y guardada (12.2 cm aprox.)
seq_0001_00269.jpeg medida y guardada (13.1 cm aprox.)
seq_0001_00270.jpeg medida y guardada (10.7 cm aprox.)
seq_0001_00271.jpeg medida y guardada (9.8 cm aprox.)
seq_0001_00301.jpeg medida y guardada (9.8 cm aprox.)
seq_0001_00326.jpeg medida y guardada (9.8 cm aprox.)
seq_0